# Task 1: 질문 유형 분류기 (Question Classification)

충남대학교 Campus ChatBot — 질문을 5개 카테고리로 분류

| Label | 카테고리 |
|-------|----------|
| 0 | 졸업요건 |
| 1 | 학교 공지사항 |
| 2 | 학사일정 |
| 3 | 식단 안내 |
| 4 | 통학/셔틀 버스 |

- 모델: `klue/bert-base` 파인튜닝
- 데이터: `data/train_augmented.json` (11,000건), `data/valid.json` (80건)
- 평가: F1 Score (macro)
- 입력: `data/test_cls.json`
- 출력: `outputs/cls_output.json`

## 1. 환경 설정

Colab에서 실행 시 Google Drive를 마운트하거나, 이 노트북이 있는 프로젝트 루트에서 실행한다.

In [ ]:
%%time
!pip install -q transformers datasets accelerate scikit-learn

# Colab 사용 시 프로젝트 루트로 이동 (Drive 마운트 후)
import os

# Google Drive 마운트 (Colab 환경에서만 실행)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/cnu_qa_system"  # Drive 내 프로젝트 경로
    os.chdir(PROJECT_ROOT)
    print(f"Colab 환경 — 작업 디렉터리: {PROJECT_ROOT}")
except ImportError:
    print(f"로컬 환경 — 작업 디렉터리: {os.getcwd()}")

In [ ]:
import json
import os
import random
from collections import Counter

import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import f1_score, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2. 설정

In [ ]:
# ── 모델 설정 ──
MODEL_NAME = "klue/bert-base"
NUM_LABELS = 5
MAX_LENGTH = 128

# ── 학습 설정 ──
EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1

# ── 경로 설정 ──
TRAIN_PATH = "data/train_augmented.json"
VALID_PATH = "data/valid.json"
TEST_PATH = "data/test_cls.json"
OUTPUT_DIR = "outputs"
MODEL_SAVE_DIR = "model/classifier"

LABEL_NAMES = ["졸업요건", "학교 공지사항", "학사일정", "식단 안내", "통학/셔틀 버스"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

## 3. 데이터 로드 & 분포 확인

In [ ]:
def load_json(path: str) -> list:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


train_data = load_json(TRAIN_PATH)
valid_data = load_json(VALID_PATH)

print(f"학습 데이터: {len(train_data)}건")
print(f"검증 데이터: {len(valid_data)}건")

# 라벨 분포 확인
for split_name, split_data in [("train", train_data), ("valid", valid_data)]:
    counts = Counter(d["label"] for d in split_data)
    print(f"\n라벨 분포 ({split_name}):")
    for label_id in sorted(counts.keys()):
        print(f"  {label_id} ({LABEL_NAMES[label_id]}): {counts[label_id]}건")

## 4. 토크나이저 & 데이터셋

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class QuestionDataset(Dataset):
    def __init__(self, data: list, tokenizer, max_length: int = MAX_LENGTH):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> dict:
        item = self.data[idx]
        encoding = self.tokenizer(
            item["question"],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        result = {k: v.squeeze(0) for k, v in encoding.items()}
        if "label" in item:
            result["labels"] = torch.tensor(item["label"], dtype=torch.long)
        return result


train_dataset = QuestionDataset(train_data, tokenizer)
valid_dataset = QuestionDataset(valid_data, tokenizer)

print(f"Train: {len(train_dataset)}, Valid: {len(valid_dataset)}")

## 5. 모델 로드 & 학습

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)
model.to(DEVICE)
print(f"모델 파라미터: {sum(p.numel() for p in model.parameters()):,}")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1_macro = f1_score(labels, predictions, average="macro")
    f1_weighted = f1_score(labels, predictions, average="weighted")
    return {"f1_macro": f1_macro, "f1_weighted": f1_weighted}


training_args = TrainingArguments(
    output_dir=MODEL_SAVE_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=10,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
)

print("학습 시작...")
trainer.train()
print("학습 완료!")

## 6. 검증 평가 & 모델 저장

In [ ]:
eval_results = trainer.evaluate()
print(f"Validation F1 (macro):    {eval_results['eval_f1_macro']:.4f}")
print(f"Validation F1 (weighted): {eval_results['eval_f1_weighted']:.4f}")

# 상세 분류 리포트
predictions = trainer.predict(valid_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = np.array([d["label"] for d in valid_data])

print("\n분류 리포트:")
print(classification_report(true_labels, pred_labels, target_names=LABEL_NAMES, digits=4))

# 모델 저장
trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print(f"\n모델 저장 완료: {MODEL_SAVE_DIR}")

## 6-1. Hugging Face Hub에 모델 업로드

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login, HfApi

# Hugging Face 로그인 (Colab: 왼쪽 🔑 아이콘 → HF_TOKEN 추가)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = input("Hugging Face 토큰을 입력하세요: ")

login(token=hf_token)

# ── 여기에 본인 HF 사용자명 입력 ──
HF_USERNAME = "your-hf-username"  # TODO: 본인 HF 아이디로 변경
REPO_NAME = f"{HF_USERNAME}/cnu-question-classifier"

api = HfApi()
api.create_repo(repo_id=REPO_NAME, exist_ok=True)
api.upload_folder(
    folder_path=MODEL_SAVE_DIR,
    repo_id=REPO_NAME,
    commit_message="Upload trained question classifier (klue/bert-base, 10k data)",
)
print(f"\n✅ 업로드 완료: https://huggingface.co/{REPO_NAME}")

## 7. 테스트셋 추론 → cls_output.json 생성

학습이 끝난 뒤, 저장된 모델을 로드하여 테스트셋을 추론한다.
평가 시에는 이 셀부터 실행해도 된다.

In [ ]:
# 저장된 모델 로드
cls_model = AutoModelForSequenceClassification.from_pretrained(MODEL_SAVE_DIR)
cls_tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_DIR)
cls_model.to(DEVICE)
cls_model.eval()
print("분류 모델 로드 완료")

In [ ]:
def classify_questions(
    questions: list[str], model, tokenizer, device, batch_size: int = 32
) -> list[int]:
    """질문 리스트를 분류하여 라벨을 반환한다."""
    model.eval()
    all_preds = []
    for i in range(0, len(questions), batch_size):
        batch_questions = questions[i : i + batch_size]
        encoding = tokenizer(
            batch_questions,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            outputs = model(**encoding)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().tolist())
    return all_preds


# 테스트 데이터 로드 & 추론
test_data = load_json(TEST_PATH)
test_questions = [d["question"] for d in test_data]
print(f"테스트 데이터: {len(test_questions)}건")

pred_labels = classify_questions(test_questions, cls_model, cls_tokenizer, DEVICE)

# 결과 생성 & 저장
cls_output = []
for question, label in zip(test_questions, pred_labels):
    cls_output.append({"question": question, "label": label})
    print(f"  [{label}] ({LABEL_NAMES[label]:8s}) {question}")

output_path = os.path.join(OUTPUT_DIR, "cls_output.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(cls_output, f, ensure_ascii=False, indent=2)

print(f"\n결과 저장 완료: {output_path}")

## 8. 결과 검증

In [ ]:
with open(output_path, encoding="utf-8") as f:
    result = json.load(f)

print(f"cls_output.json: {len(result)}건")
print(f"샘플:\n{json.dumps(result[:3], ensure_ascii=False, indent=2)}")

pred_dist = Counter(d["label"] for d in result)
print("\n예측 라벨 분포:")
for label_id in sorted(pred_dist.keys()):
    print(f"  {label_id} ({LABEL_NAMES[label_id]}): {pred_dist[label_id]}건")

print("\n모든 라벨이 0~4 범위인지:", all(0 <= d["label"] <= 4 for d in result))

## 완료

- `model/classifier/` — 학습된 분류 모델
- `outputs/cls_output.json` — 테스트셋 분류 결과